# 02 - Modeling & Evaluation

## AI-Generated Text Detection Project

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             classification_report, roc_curve)
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Data

In [ ]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')
test_df = pd.read_csv('../data/processed/test.csv')

X_train, y_train = train_df['text'], train_df['label']
X_val, y_val = val_df['text'], val_df['label']
X_test, y_test = test_df['text'], test_df['label']

print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

## 2. Feature Engineering - TF-IDF

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    stop_words='english',
    analyzer='word'
)

X_train_tfidf = tfidf.fit_transform(X_train.astype(str))
X_val_tfidf = tfidf.transform(X_val.astype(str))
X_test_tfidf = tfidf.transform(X_test.astype(str))

print(f'TF-IDF features: {X_train_tfidf.shape[1]}')

In [ ]:
char_tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(2, 5),
    min_df=2,
    max_df=0.95,
    analyzer='char'
)

X_train_char = char_tfidf.fit_transform(X_train.astype(str))
X_val_char = char_tfidf.transform(X_val.astype(str))
X_test_char = char_tfidf.transform(X_test.astype(str))

print(f'Character n-gram features: {X_train_char.shape[1]}')

In [ ]:
from scipy.sparse import hstack

X_train_combined = hstack([X_train_tfidf, X_train_char])
X_val_combined = hstack([X_val_tfidf, X_val_char])
X_test_combined = hstack([X_test_tfidf, X_test_char])

print(f'Combined features: {X_train_combined.shape[1]}')

## 3. Baseline Model - Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(X_train_combined, y_train)

In [ ]:
y_pred_lr = lr_model.predict(X_val_combined)
y_prob_lr = lr_model.predict_proba(X_val_combined)[:, 1]

print('Logistic Regression - Validation Results')
print('='*50)
print(f'Accuracy: {accuracy_score(y_val, y_pred_lr):.4f}')
print(f'Precision: {precision_score(y_val, y_pred_lr):.4f}')
print(f'Recall: {recall_score(y_val, y_pred_lr):.4f}')
print(f'F1-Score: {f1_score(y_val, y_pred_lr):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_val, y_prob_lr):.4f}')

## 4. XGBoost Model

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

xgb_model.fit(X_train_combined, y_train)

In [ ]:
y_pred_xgb = xgb_model.predict(X_val_combined)
y_prob_xgb = xgb_model.predict_proba(X_val_combined)[:, 1]

print('XGBoost - Validation Results')
print('='*50)
print(f'Accuracy: {accuracy_score(y_val, y_pred_xgb):.4f}')
print(f'Precision: {precision_score(y_val, y_pred_xgb):.4f}')
print(f'Recall: {recall_score(y_val, y_pred_xgb):.4f}')
print(f'F1-Score: {f1_score(y_val, y_pred_xgb):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_val, y_prob_xgb):.4f}')

## 5. Hyperparameter Tuning (XGBoost)

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

xgb_base = xgb.XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')

grid_search = GridSearchCV(
    xgb_base,
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_combined, y_train)

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best F1 score: {grid_search.best_score_:.4f}')

In [ ]:
best_xgb = grid_search.best_estimator_

## 6. Final Evaluation on Test Set

In [ ]:
y_pred_final = best_xgb.predict(X_test_combined)
y_prob_final = best_xgb.predict_proba(X_test_combined)[:, 1]

print('XGBoost (Tuned) - Test Results')
print('='*50)
print(f'Accuracy: {accuracy_score(y_test, y_pred_final):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_final):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_final):.4f}')
print(f'F1-Score: {f1_score(y_test, y_pred_final):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_final):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_final, target_names=['Human', 'AI']))

## 7. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_final)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Human', 'AI'], yticklabels=['Human', 'AI'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix - XGBoost')
plt.tight_layout()
plt.savefig('../app/assets/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob_final)
auc_score = roc_auc_score(y_test, y_prob_final)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, label=f'XGBoost (AUC = {auc_score:.4f})', color='steelblue', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../app/assets/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Model Comparison

In [ ]:
results = {
    'Model': ['Logistic Regression', 'XGBoost (Default)', 'XGBoost (Tuned)'],
    'Accuracy': [
        accuracy_score(y_val, y_pred_lr),
        accuracy_score(y_val, y_pred_xgb),
        accuracy_score(y_test, y_pred_final)
    ],
    'Precision': [
        precision_score(y_val, y_pred_lr),
        precision_score(y_val, y_pred_xgb),
        precision_score(y_test, y_pred_final)
    ],
    'Recall': [
        recall_score(y_val, y_pred_lr),
        recall_score(y_val, y_pred_xgb),
        recall_score(y_test, y_pred_final)
    ],
    'F1-Score': [
        f1_score(y_val, y_pred_lr),
        f1_score(y_val, y_pred_xgb),
        f1_score(y_test, y_pred_final)
    ],
    'ROC-AUC': [
        roc_auc_score(y_val, y_prob_lr),
        roc_auc_score(y_val, y_prob_xgb),
        roc_auc_score(y_test, y_prob_final)
    ]
}

results_df = pd.DataFrame(results)
results_df.to_csv('../reports/model_comparison.csv', index=False)
print(results_df)

## 10. Save Models

In [ ]:
joblib.dump(best_xgb, '../models/best_model.pkl')
joblib.dump({'word_tfidf': tfidf, 'char_tfidf': char_tfidf}, '../models/preprocessing.pkl')

print('Models saved to models/')